# 04. Activity Contribution Analysis

This notebook computes the percentage contribution of Combat, Collection, and Exploration activities for each telemetry window based on normalized features.

## 1. Setup and Imports

In [9]:
import pandas as pd
import os
import numpy as np

# Inputs & Outputs
INPUT_PATH = 'data/processed/3_normalized_telemetry.csv'
OUTPUT_PATH = 'data/processed/4_activity_contributions.csv'

print("Libraries imported.")

Libraries imported.


## 2. Load Data

In [10]:
print("Loading normalized telemetry data...")
if os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(df)} rows.")
else:
    raise FileNotFoundError(f"File not found at {INPUT_PATH}")

Loading normalized telemetry data...
Loaded 2838 rows.


## 3. Define Feature Categories
We group features into three main gameplay activities: Combat, Collection, and Exploration.

In [11]:
# Define Categories
# Using both top-level and rawJson keys if present to catch all variants
F_COMBAT = ['enemiesHit', 'damageDone', 'timeInCombat', 'kills', 
            'rawJson.enemies_hit', 'rawJson.damage_done', 'rawJson.time_in_combat', 'rawJson.kills']
F_COLLECT = ['itemsCollected', 'pickupAttempts', 'timeNearInteractables',
             'rawJson.items_collected', 'rawJson.pickup_attempts', 'rawJson.time_near_interactables']
F_EXPLORE = ['distanceTraveled', 'timeSprinting', 'timeOutOfCombat',
             'rawJson.distance_traveled', 'rawJson.time_out_of_combat', 'rawJson.time_sprinting']

print("Feature categories defined.")

Feature categories defined.


In [12]:
# Filter to cols actually in DF
cols_combat = [c for c in F_COMBAT if c in df.columns]
cols_collect = [c for c in F_COLLECT if c in df.columns]
cols_explore = [c for c in F_EXPLORE if c in df.columns]

print(f"Combat Features Found: {cols_combat}")
print(f"Collection Features Found: {cols_collect}")
print(f"Exploration Features Found: {cols_explore}")

Combat Features Found: ['enemiesHit', 'damageDone', 'timeInCombat', 'kills', 'rawJson.enemies_hit', 'rawJson.damage_done', 'rawJson.time_in_combat', 'rawJson.kills']
Collection Features Found: ['itemsCollected', 'pickupAttempts', 'timeNearInteractables', 'rawJson.items_collected', 'rawJson.pickup_attempts', 'rawJson.time_near_interactables']
Exploration Features Found: ['distanceTraveled', 'timeSprinting', 'timeOutOfCombat', 'rawJson.distance_traveled', 'rawJson.time_out_of_combat', 'rawJson.time_sprinting']


## 4. Compute Raw Scores
Sum the normalized features for each category.

In [13]:
# Compute Raw Scores (Sum of normalized features)
df['score_combat'] = df[cols_combat].sum(axis=1)
df['score_collect'] = df[cols_collect].sum(axis=1)
df['score_explore'] = df[cols_explore].sum(axis=1)

# Compute Total Score
df['score_total'] = df['score_combat'] + df['score_collect'] + df['score_explore']

print("Raw scores computed.")

Raw scores computed.


## 5. Compute Contribution Percentages
Calculate the share of each activity relative to the total activity score in that window.

In [14]:
# Compute Percentages
# Avoid Division by Zero
mask_nonzero = df['score_total'] > 0

df['pct_combat'] = 0.0
df['pct_collect'] = 0.0
df['pct_explore'] = 0.0

df.loc[mask_nonzero, 'pct_combat'] = df.loc[mask_nonzero, 'score_combat'] / df.loc[mask_nonzero, 'score_total']
df.loc[mask_nonzero, 'pct_collect'] = df.loc[mask_nonzero, 'score_collect'] / df.loc[mask_nonzero, 'score_total']
df.loc[mask_nonzero, 'pct_explore'] = df.loc[mask_nonzero, 'score_explore'] / df.loc[mask_nonzero, 'score_total']

# Verification Stats
print("\n--- Percentage Stats ---")
print(df[['pct_combat', 'pct_collect', 'pct_explore']].describe())


--- Percentage Stats ---
        pct_combat  pct_collect  pct_explore
count  2838.000000  2838.000000  2838.000000
mean      0.012087     0.001680     0.986232
std       0.034594     0.011711     0.036359
min       0.000000     0.000000     0.037132
25%       0.000000     0.000000     0.983165
50%       0.000627     0.000779     0.996069
75%       0.015749     0.001689     0.999120
max       0.962868     0.395072     1.000000


## 6. Compute Deltas (Activity Change Over Time)
We calculate how much the percentages change from one window to the next for each user.

In [15]:
print("Preparing to compute deltas...")

# Robustness: Drop rows without userId
if df['userId'].isna().any():
    print("Warning: Found rows with missing userId. Dropping them.")
    df = df.dropna(subset=['userId'])

# Ensure Strict Sorting by User and Time before computing lag
t_col = 'timestamp' if 'timestamp' in df.columns else 'time'
if t_col in df.columns:
    print(f"Sorting data by userId and {t_col}...")
    # Helper for sort
    df['sort_helper'] = pd.to_numeric(df[t_col], errors='coerce')
    # Fallback to datetime if numeric fails
    if df['sort_helper'].isna().sum() > len(df) * 0.5:
         print("Timestamps appear to be non-numeric, parsing as datetime...")
         df['sort_helper'] = pd.to_datetime(df[t_col], errors='coerce')
    
    df.sort_values(by=['userId', 'sort_helper'], inplace=True)
    df.drop(columns=['sort_helper'], inplace=True)
    print("Sorting complete.")

print("Calculating Diff (Group By User)...")
# Calculate Diff (Group by userId to ensure lag is per-player)
cols_pct = ['pct_combat', 'pct_collect', 'pct_explore']
try:
    df[['delta_combat', 'delta_collect', 'delta_explore']] = (
        df.groupby('userId')[cols_pct].diff().fillna(0)
    )
    print("Delta calculation successful.")
except Exception as e:
    print(f"Error calculating deltas: {e}")

# Verification for Deltas
print("\n--- Delta Stats ---")
print(df[['delta_combat', 'delta_collect', 'delta_explore']].describe())

Preparing to compute deltas...
Sorting data by userId and timestamp...
Timestamps appear to be non-numeric, parsing as datetime...
Sorting complete.
Calculating Diff (Group By User)...
Delta calculation successful.

--- Delta Stats ---
       delta_combat  delta_collect  delta_explore
count   2838.000000   2.838000e+03    2838.000000
mean       0.000093   9.647411e-07      -0.000094
std        0.047192   1.654846e-02       0.049857
min       -0.930835  -3.917202e-01      -0.948139
25%       -0.007455  -7.450436e-04      -0.007719
50%        0.000000   0.000000e+00       0.000000
75%        0.007355   7.203519e-04       0.007446
max        0.948139   3.946099e-01       0.930835


## 7. Export Data

In [16]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved activity contributions to {OUTPUT_PATH}")
print("Activity contribution analysis complete.")


Saved activity contributions to data/processed/4_activity_contributions.csv
Activity contribution analysis complete.
